In [1]:
%cd /content
!rm -rf mccain-internship
!git clone https://github.com/PushkargithubCSE/mccain-internship.git
%cd mccain-internship/week-3/potato-finetuning
PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"

!pip install -q transformers roboflow python-dotenv huggingface_hub pycocotools

/content
Cloning into 'mccain-internship'...
remote: Enumerating objects: 566, done.
remote: Counting objects: 100% (566/566), done.
remote: Compressing objects: 100% (413/413), done.
remote: Total 566 (delta 147), reused 508 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (566/566), 24.27 MiB | 20.27 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/mccain-internship/week-3/potato-finetuning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 143.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.5 MB/s eta 0:00:00


In [2]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv(f"{PROJECT_PATH}/.env")
api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("ROBOFLOW_API_KEY is missing from the project .env file")

rf = Roboflow(api_key=api_key)
project = rf.workspace("pushkar-chandra").project("segmentation-jajrd")
dataset = project.version(2).download("coco-segmentation")
print(f"Dataset ready at: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Segmentation-2 in coco-segmentation:: 100%|██████████| 20/20 [00:00<00:00, 3027.94it/s]

✓ Dataset ready at: /content/mccain-internship/week-3/potato-finetuning/Segmentation-2


In [3]:
import os
from transformers import Sam3Model, Sam3Processor
from huggingface_hub import login

hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise RuntimeError("HUGGINGFACE_TOKEN is missing from the project .env file")

login(token=hf_token)

model = Sam3Model.from_pretrained("facebook/sam3")
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("SAM-3 loaded")

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

✓ SAM-3 loaded


In [4]:
for param in model.parameters():
    param.requires_grad = False

trainable_params = 0
for name, param in model.named_parameters():
    if "decoder" in name.lower():
        param.requires_grad = True
        trainable_params += param.numel()

total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M ({100*trainable_params/total_params:.2f}%)")

if trainable_params == 0:
    print("\n⚠️ No 'decoder' match — listing all layer names:")
    for name, _ in model.named_parameters():
        print(name)

Trainable: 13.87M / 840.38M (1.65%)


In [5]:
import json
import numpy as np
from PIL import Image
from pycocotools import mask as coco_mask
from torch.utils.data import Dataset
import torch

class SAM3SegDataset(Dataset):
    def __init__(self, coco_json_path, images_dir):
        with open(coco_json_path) as f:
            self.coco = json.load(f)
        self.images_dir = images_dir
        self.img_id_to_info = {img['id']: img for img in self.coco['images']}
        self.annotations = self.coco['annotations']

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_info = self.img_id_to_info[ann['image_id']]
        img_path = f"{self.images_dir}/{img_info['file_name']}"
        image = Image.open(img_path).convert("RGB")

        h, w = img_info['height'], img_info['width']
        seg = ann['segmentation']

        if isinstance(seg, dict):
            # Already RLE format (has 'counts' and 'size' keys)
            if isinstance(seg['counts'], list):
                # uncompressed RLE -> compress it first
                rle = coco_mask.frPyObjects(seg, h, w)
            else:
                # already compressed RLE (string counts)
                rle = seg
            binary_mask = coco_mask.decode(rle)
        else:
            # Polygon format (list of lists)
            rles = coco_mask.frPyObjects(seg, h, w)
            rle = coco_mask.merge(rles) if isinstance(rles, list) else rles
            binary_mask = coco_mask.decode(rle)

        if binary_mask.ndim == 3:
            binary_mask = binary_mask.any(axis=2).astype(np.uint8)

        return image, torch.from_numpy(binary_mask).float(), "damage"

train_ds = SAM3SegDataset(
    f"{dataset.location}/train/_annotations.coco.json",
    f"{dataset.location}/train"
)
print(f"✓ Train dataset: {len(train_ds)} samples")
img, mask, prompt = train_ds[0]
print("Image size:", img.size, "| Mask shape:", mask.shape, "| Mask sum (pixels marked):", mask.sum().item())

✓ Train dataset: 10 samples
Image size: (432, 432) | Mask shape: torch.Size([432, 432]) | Mask sum (pixels marked): 4943.0


In [6]:
image, gt_mask, prompt = train_ds[0]
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

print("Output type:", type(outputs))
print("Output attributes:", outputs.keys() if hasattr(outputs, 'keys') else dir(outputs))
print()
print("pred_masks shape:", outputs.pred_masks.shape)

# Check for confidence/score fields that would tell us which of the 200 to pick
for attr in ['iou_scores', 'pred_scores', 'scores', 'logits', 'pred_boxes', 'objectness_logits']:
    if hasattr(outputs, attr):
        val = getattr(outputs, attr)
        print(f"{attr}: shape = {val.shape if hasattr(val, 'shape') else val}")

NameError: name 'device' is not defined

In [7]:
print("pred_logits shape:", outputs.pred_logits.shape)
print("pred_logits sample values:", outputs.pred_logits[0, :5])
print()
print("presence_logits shape:", outputs.presence_logits.shape)
print("presence_logits values:", outputs.presence_logits)
print()
print("semantic_seg shape:", outputs.semantic_seg.shape if outputs.semantic_seg is not None else None)

In [7]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT available. Using CPU.")


GPU is available: Tesla T4


In [8]:
import os
from huggingface_hub import login, whoami

hf_token_write = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token_write:
    raise RuntimeError("HUGGINGFACE_TOKEN is missing from the project .env file")

login(token=hf_token_write)

user_info = whoami()
print("Authenticated as:", user_info["name"])

✓ Authenticated as: pushkar0002


In [9]:
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, create_repo

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

EPOCHS = 20
OUTPUT_DIR = f"{PROJECT_PATH}/sam3/checkpoints/sam3_run1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    total_loss = 0
    for image, gt_mask, prompt in train_ds:
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
        gt_mask_batched = gt_mask.unsqueeze(0).unsqueeze(0).to(device)  # [1, 1, H, W]

        outputs = model(**inputs)

        # Select the highest-confidence query out of 200
        scores = outputs.pred_logits[0]  # [200]
        best_idx = scores.argmax()

        pred_mask = outputs.pred_masks[0, best_idx].unsqueeze(0).unsqueeze(0)  # [1, 1, h, w]

        # Resize predicted mask to match ground truth resolution
        pred_mask_resized = F.interpolate(pred_mask, size=gt_mask.shape[-2:], mode="bilinear")

        loss = F.binary_cross_entropy_with_logits(pred_mask_resized.squeeze(1), gt_mask_batched.squeeze(1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} — avg loss: {total_loss/len(train_ds):.4f}")

print("✓ Training complete")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
  
REPO_ID = "pushkar0002/finetune"
api = HfApi()
create_repo(repo_id=REPO_ID, exist_ok=True, repo_type="model")
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, repo_type="model")
print(f"✓ Uploaded to: https://huggingface.co/{REPO_ID}")

Epoch 1/20 — avg loss: 0.0796
Epoch 2/20 — avg loss: 0.0271
Epoch 3/20 — avg loss: 0.0233
Epoch 4/20 — avg loss: 0.0179
Epoch 5/20 — avg loss: 0.0159
Epoch 6/20 — avg loss: 0.0144
Epoch 7/20 — avg loss: 0.0131
Epoch 8/20 — avg loss: 0.0124
Epoch 9/20 — avg loss: 0.0111
Epoch 10/20 — avg loss: 0.0103
Epoch 11/20 — avg loss: 0.0097
Epoch 12/20 — avg loss: 0.0091
Epoch 13/20 — avg loss: 0.0090
Epoch 14/20 — avg loss: 0.0089
Epoch 15/20 — avg loss: 0.0091
Epoch 16/20 — avg loss: 0.0098
Epoch 17/20 — avg loss: 0.0094
Epoch 18/20 — avg loss: 0.0091
Epoch 19/20 — avg loss: 0.0073
Epoch 20/20 — avg loss: 0.0068
✓ Training complete


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Uploaded to: https://huggingface.co/pushkar0002/finetune
